In [ ]:
# ==========================================
# CELL 1: Setup, Installs & Imports
# ==========================================

!pip install -q h5py gdown e2cnn

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision import models
from torchvision.datasets import PCAM
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm import tqdm
import numpy as np
import h5py

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# CONFIGURATION
FAST_DEBUG_MODE = False
BATCH_SIZE = 32
EPOCHS = 5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.3/225.3 kB 11.6 MB/s eta 0:00:00
Using device: cuda


In [ ]:
# ==========================================
# CELL 1.5: DRIVE MOUNT & DATA SYNC
# ==========================================
from google.colab import drive
import os
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define permanent storage on your Drive
drive_path = '/content/drive/MyDrive/Erdos_Project/PCAM_Data'
os.makedirs(drive_path, exist_ok=True)

# 3. Define the local Colab path (where PyTorch expects it)
local_path = './data/pcam'
os.makedirs(local_path, exist_ok=True)

# 4. Define the Zenodo files
zenodo_base = "https://zenodo.org/records/2546921/files/"
files = [
    "camelyonpatch_level_2_split_train_x.h5.gz",
    "camelyonpatch_level_2_split_train_y.h5.gz",
    "camelyonpatch_level_2_split_test_x.h5.gz",
    "camelyonpatch_level_2_split_test_y.h5.gz",
    "camelyonpatch_level_2_split_valid_x.h5.gz",
    "camelyonpatch_level_2_split_valid_y.h5.gz"
]

print("Checking Data Sync...")
for f in files:
    drive_file = os.path.join(drive_path, f)
    local_file = os.path.join(local_path, f)

    # STEP A: If it's not on Drive, download from Zenodo to Drive
    if not os.path.exists(drive_file):
        print(f"Downloading {f} to Google Drive (This only happens once)...")
        os.system(f"wget -q -c {zenodo_base}{f} -O {drive_file}")

    # STEP B: Copy from Drive to Local Colab SSD for fast training
    if not os.path.exists(local_file):
        print(f"Copying {f} to fast local storage...")
        shutil.copy(drive_file, local_file)

print("\nData is synced to local storage. Ready for PyTorch!")

Mounted at /content/drive
Checking Data Sync...
Copying camelyonpatch_level_2_split_train_x.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_train_y.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_test_x.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_test_y.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_valid_x.h5.gz to fast local storage...
Copying camelyonpatch_level_2_split_valid_y.h5.gz to fast local storage...

Data is synced to local storage. Ready for PyTorch!


In [ ]:
# ==========================================
# CELL 1.7: EXTRACT COMPRESSED FILES
# ==========================================
print("Extracting .gz files into raw .h5 files...")
!gunzip -f ./data/pcam/*.gz
print("Extraction complete! PyTorch will now recognize the dataset.")

Extracting .gz files into raw .h5 files...
Extraction complete! PyTorch will now recognize the dataset.


In [ ]:
# ==========================================
# CELL 1.9: CALCULATE EXACT DATASET STATISTICS
# ==========================================
import torch
from torch.utils.data import DataLoader
from torchvision.datasets.pcam import PCAM
import torchvision.transforms as T
from tqdm import tqdm

# 1. Load the raw dataset (Strictly ToTensor, no augmentations!)
raw_transform = T.ToTensor()
raw_train_ds = PCAM(root='./data', split='train', download=False, transform=raw_transform)

# 2. Use a large batch size to speed up the loop
calc_loader = DataLoader(raw_train_ds, batch_size=256, shuffle=False, num_workers=2)

print("Calculating exact Mean and Standard Deviation...")

# Initialize placeholders for the moments
channels_sum = torch.zeros(3)
channels_squared_sum = torch.zeros(3)
num_batches = 0

loop = tqdm(calc_loader, desc="Computing Stats")
for images, _ in loop:
    # Average across batch, height, and width (leaving only the 3 color channels)
    channels_sum += torch.mean(images, dim=[0, 2, 3])
    channels_squared_sum += torch.mean(images**2, dim=[0, 2, 3])
    num_batches += 1

# Calculate final expected values
mean = channels_sum / num_batches
std = (channels_squared_sum / num_batches - mean ** 2) ** 0.5

print("\n\n>>> EXACT PCAM STATISTICS <<<")
print(f"pcam_mean = {[round(x.item(), 3) for x in mean]}")
print(f"pcam_std  = {[round(x.item(), 3) for x in std]}")

Calculating exact Mean and Standard Deviation...


Computing Stats:  30%|███       | 310/1024 [01:08<02:42,  4.40it/s]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x792d269b7ec0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Computing Stats:  31%|███       | 314/1024 [01:09<02:42,  4.37it/s]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x792d269b7ec0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 



>>> EXACT PCAM STATISTICS <<<
pcam_mean = [0.701, 0.538, 0.692]
pcam_std  = [0.235, 0.277, 0.213]


In [ ]:
# ==========================================
# CELL 2: FULL D4 TEST PIPELINE
# ==========================================

# 1. Restore the exact PCam Normalization constants
pcam_mean = [0.701, 0.538, 0.692]
pcam_std  = [0.235, 0.277, 0.213]
normalize = T.Normalize(mean=pcam_mean, std=pcam_std)

def get_test_transform(angle, reflect=False):
    """
    Deterministically rotate and optionally reflect the image.
    Use pure matrix transpositions to guarantee zero data loss.
    """
    k_map = {0: 0, 90: 1, 180: 2, 270: 3}
    k = k_map[angle]

    transforms_list = [T.ToTensor()]

    # 2. Apply Rotation (Dims -2 and -1 are Height and Width)
    if k != 0:
        transforms_list.append(T.Lambda(lambda x: torch.rot90(x, k=k, dims=[-2, -1])))

    # 3. Apply Reflection (Horizontal flip along the Width axis, dim -1)
    if reflect:
        transforms_list.append(T.Lambda(lambda x: torch.flip(x, dims=[-1])))

    # 4. Apply the Normalization
    transforms_list.append(normalize)

    return T.Compose(transforms_list)

print("Building Full D4 Test DataLoaders (8 States)...")

# Create the 8 distinct test sets programmatically
test_loaders = {}
for angle in [0, 90, 180, 270]:
    # A. The Pure Rotations
    ds_rot = PCAM(root='./data', split='test', download=False, transform=get_test_transform(angle, reflect=False))
    test_loaders[f"{angle}_deg"] = DataLoader(ds_rot, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # B. The Reflections
    ds_ref = PCAM(root='./data', split='test', download=False, transform=get_test_transform(angle, reflect=True))
    test_loaders[f"{angle}_deg_flip"] = DataLoader(ds_ref, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Successfully loaded {len(test_loaders)} specific rotational/reflection states.")

Building Full D4 Test DataLoaders (8 States)...
Successfully loaded 8 specific rotational/reflection states.


In [ ]:
# ==========================================
# CELL 3: ARCHITECTURE & DYNAMIC TRAINING LOOP
# ==========================================

def get_model(model_name="resnet18"):
    """Loads a pre-trained ResNet and modifies the head for 2-class PCam."""
    if model_name == "resnet18":
        # Load architecture with ImageNet weights to give the model a 'head start'
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    elif model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    # Replace final layer (1000 ImageNet classes -> 2 PCam classes)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2)
    return model.to(device)

def train_model(model, loader, epochs):
    criterion = nn.CrossEntropyLoss()
    # Initial learning rate set to 0.001
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # DYNAMIC LEARNING RATE: Reduces LR by half if the loss doesn't improve for 1 epoch
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch_idx, (images, labels) in enumerate(loop):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Track Average Loss for the entire epoch
            running_loss += loss.item()
            avg_loss = running_loss / (batch_idx + 1)

            # Update progress bar with clinical-grade tracking
            loop.set_postfix(
                avg_loss=f"{avg_loss:.4f}",
                lr=f"{optimizer.param_groups[0]['lr']:.6e}"
            )

        # At the end of each epoch, tell the scheduler the average loss
        scheduler.step(avg_loss)

    return model

In [ ]:
# ==========================================
# CELL 4: EVALUATION & METRICS
# ==========================================

def evaluate_robustness(model, test_loaders):
    model.eval()
    results = {}
    base_predictions = None

    with torch.no_grad():
        for angle_name, loader in test_loaders.items():
            all_preds, all_labels, all_probs = [], [], []

            loop = tqdm(loader, desc=f"Testing {angle_name}")
            for images, labels in loop:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)

                probs = torch.softmax(outputs, dim=1)[:, 1]
                _, preds = torch.max(outputs, 1)

                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            acc = accuracy_score(all_labels, all_preds)
            auc = roc_auc_score(all_labels, all_probs)

            # Calculate how many predictions flipped compared to 0 degrees
            flip_rate = 0.0
            if angle_name == "0_deg":
                base_predictions = np.array(all_preds)
            else:
                current_predictions = np.array(all_preds)
                flips = np.sum(current_predictions != base_predictions)
                flip_rate = (flips / len(base_predictions)) * 100

            results[angle_name] = {"Accuracy": acc, "AUC": auc, "Flip_Rate_%": flip_rate}

    return results

def print_results(model_name, results):
    print(f"\n{'='*45}")
    print(f"RESULTS FOR {model_name.upper()}")
    print(f"{'='*45}")
    print(f"{'Angle':<10} | {'Accuracy':<10} | {'AUC':<10} | {'Flip Rate':<10}")
    print("-" * 45)
    for angle, metrics in results.items():
        print(f"{angle:<10} | {metrics['Accuracy']:.4f}   | {metrics['AUC']:.4f}   | {metrics['Flip_Rate_%']:.2f}%")

In [ ]:
# ==========================================
# CELL 5: RUN THE EXPERIMENT
# ==========================================

print("\n>>> STARTING RESNET-18 BASELINE EXPERIMENT <<<")
resnet18 = get_model("resnet18")
resnet18 = train_model(resnet18, train_loader, EPOCHS)
res18_results = evaluate_robustness(resnet18, test_loaders)
print_results("ResNet-18", res18_results)


>>> STARTING RESNET-18 BASELINE EXPERIMENT <<<
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 213MB/s]
Testing 270_deg: 100%|██████████| 1024/1024 [01:07<00:00, 15.17it/s]


RESULTS FOR RESNET-18
Angle      | Accuracy   | AUC        | Flip Rate 
---------------------------------------------
0_deg      | 0.8490   | 0.9317   | 0.00%
90_deg     | 0.8157   | 0.9185   | 10.97%
180_deg    | 0.8043   | 0.9091   | 11.30%
270_deg    | 0.8351   | 0.9227   | 10.38%


In [ ]:
# ==========================================
# CELL 5.5: SAVE MODEL WEIGHTS TO DRIVE
# ==========================================
import os
import torch

# 1. Define the path on your permanently mounted Google Drive
save_dir = '/content/drive/MyDrive/Erdos_Project/Models'
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'resnet18_baseline.pth')

# 2. Save the model's state dictionary (the trained weights)
torch.save(resnet18.state_dict(), save_path)
print(f"Success! Model weights permanently saved to:\n{save_path}")

Success! Model weights permanently saved to:
/content/drive/MyDrive/Erdos_Project/Models/resnet18_baseline.pth


In [ ]:
# ==========================================
# FINAL RESNET-18 EVALUATION (LOAD & METRICS)
# ==========================================
import os
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, confusion_matrix, precision_recall_curve
from tqdm import tqdm




# 1. Rebuild the Skeleton
print("Rebuilding ResNet-18 skeleton...")
resnet18 = models.resnet18(weights=None) # weights=None because we are loading our own
num_ftrs = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_ftrs, 2) # Matching your 2-class PCam setup
resnet18 = resnet18.to(device)

# 2. Load your weights
save_path = '/content/drive/MyDrive/Erdos_Project/Models/resnet18_baseline.pth'
if os.path.exists(save_path):
    resnet18.load_state_dict(torch.load(save_path, map_location=device))
    resnet18.eval()
    print("Baseline brain successfully loaded!")
else:
    print(f"Error: Could not find weights at {save_path}. Check your Drive path!")

# 2. COMBINED EVALUATION LOOP
def full_resnet_evaluation(model, loaders):
    robustness_results = {}
    base_predictions = None

    # Variables specifically for 0-degree clinical metrics
    clinical_labels = []
    clinical_probs = []

    print(">>> STARTING RESNET-18 ROBUSTNESS & CLINICAL EVALUATION <<<")
    with torch.no_grad():
        for angle_name, loader in loaders.items():
            all_preds, all_labels, all_probs = [], [], []

            loop = tqdm(loader, desc=f"Testing {angle_name}")
            for images, labels in loop:
                images, labels = images.to(device), labels.to(device)

                # Forward pass
                outputs = model(images)

                # Extract probabilities and standard 50% threshold predictions
                probs = F.softmax(outputs, dim=1)[:, 1]
                _, preds = torch.max(outputs, 1)

                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            # Calculate Standard Metrics
            acc = accuracy_score(all_labels, all_preds)
            auc = roc_auc_score(all_labels, all_probs)

            # Calculate Flip Rate
            if angle_name == "0_deg":
                base_predictions = np.array(all_preds)
                flip_rate = 0.0

                # Save the 0-degree arrays for the custom clinical threshold calculation
                clinical_labels = np.array(all_labels)
                clinical_probs = np.array(all_probs)
            else:
                current_predictions = np.array(all_preds)
                flips = np.sum(current_predictions != base_predictions)
                flip_rate = (flips / len(base_predictions)) * 100

            robustness_results[angle_name] = {"Accuracy": acc, "AUC": auc, "Flip_Rate_%": flip_rate}

    # 3. CALCULATE CLINICAL RECALL (ON 0-DEGREE BASELINE)
    precisions, recalls, thresholds = precision_recall_curve(clinical_labels, clinical_probs)

    # Find the exact threshold needed to hit 95% Recall
    target_recall = 0.95
    idx = np.where(recalls >= target_recall)[0][-1]
    optimal_threshold = thresholds[idx]

    # Generate new predictions using the clinical threshold
    custom_preds = (clinical_probs >= optimal_threshold).astype(int)
    recall = recall_score(clinical_labels, custom_preds)
    tn, fp, fn, tp = confusion_matrix(clinical_labels, custom_preds).ravel()

   # 4. PRINT FINAL TABLES
    print(f"\n{'='*60}")
    print(f"RESNET-18 ROBUSTNESS RESULTS (FULL D4 MANIFOLD)")
    print(f"{'='*60}")
    # Increased spacing from <10 to <15 for the new flip labels
    print(f"{'Angle':<15} | {'Accuracy':<10} | {'AUC':<10} | {'Flip Rate':<10}")
    print("-" * 60)
    for angle, metrics in robustness_results.items():
        print(f"{angle:<15} | {metrics['Accuracy']:.4f}   | {metrics['AUC']:.4f}   | {metrics['Flip_Rate_%']:.2f}%")

    print(f"\n{'='*60}")
    print(f"RESNET-18 CLINICAL METRICS (0-DEG BASELINE)")
    print(f"Targeting 95% Recall | Optimal Threshold: {(optimal_threshold*100):.2f}%")
    print(f"{'='*60}")
    print(f"Recall (Sensitivity):            {recall:.4f} ({(recall*100):.2f}%)")
    print("------------------------------------------------------------")
    print(f"True Positives (Cancer Found):   {tp}")
    print(f"False Negatives (Cancer Missed): {fn}")
    print("------------------------------------------------------------")
    print(f"True Negatives (Healthy Found):  {tn}")
    print(f"False Positives (False Alarms):  {fp}")
    print("============================================================")

# Execute the function
full_resnet_evaluation(resnet18, test_loaders)

Rebuilding ResNet-18 skeleton...
Baseline brain successfully loaded!
>>> STARTING RESNET-18 ROBUSTNESS & CLINICAL EVALUATION <<<


Testing 270_deg_flip: 100%|██████████| 1024/1024 [00:32<00:00, 31.72it/s]



RESNET-18 ROBUSTNESS RESULTS (FULL D4 MANIFOLD)
Angle           | Accuracy   | AUC        | Flip Rate 
------------------------------------------------------------
0_deg           | 0.8490   | 0.9317   | 0.00%
0_deg_flip      | 0.8193   | 0.9171   | 10.31%
90_deg          | 0.8157   | 0.9185   | 10.96%
90_deg_flip     | 0.8362   | 0.9210   | 10.21%
180_deg         | 0.8044   | 0.9091   | 11.30%
180_deg_flip    | 0.8285   | 0.9224   | 9.51%
270_deg         | 0.8351   | 0.9228   | 10.38%
270_deg_flip    | 0.8199   | 0.9133   | 10.17%

RESNET-18 CLINICAL METRICS (0-DEG BASELINE)
Targeting 95% Recall | Optimal Threshold: 0.88%
Recall (Sensitivity):            0.9501 (95.01%)
------------------------------------------------------------
True Positives (Cancer Found):   15560
False Negatives (Cancer Missed): 817
------------------------------------------------------------
True Negatives (Healthy Found):  9634
False Positives (False Alarms):  6757


In [ ]:
# ==========================================
# CELL 8: RECALL & SPECIFICITY STABILITY AUDIT
# ==========================================
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import recall_score, confusion_matrix
from tqdm import tqdm

clinical_threshold = 0.25

print(f"\n{'='*95}")
print(f"   CLINICAL METRICS (THRESHOLD = {clinical_threshold*100}%)")
print(f"{'='*95}")
print(f"{'Angle':<15} | {'Recall':<10} | {'Missed (FN)':<15} | {'False Alarms (FP)':<18} | {'Specificity'}")
print("-" * 95)

with torch.no_grad():
    for angle_name, loader in test_loaders.items():
        all_labels = []
        all_probs = []

        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = resnet18(images)
            probs = F.softmax(outputs, dim=1)[:, 1]

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

        all_labels = np.array(all_labels)
        all_probs = np.array(all_probs)

        # Apply the 25% threshold
        custom_preds = (all_probs >= clinical_threshold).astype(int)

        # Calculate metrics for this specific angle
        recall = recall_score(all_labels, custom_preds)
        tn, fp, fn, tp = confusion_matrix(all_labels, custom_preds).ravel()

        # Calculate Specificity (True Negative Rate)
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        print(f"{angle_name:<15} | {recall:.4f}     | {fn:<15} | {fp:<18} | {specificity:.4f}")

print("=" * 95)


   CLINICAL METRICS (THRESHOLD = 25.0%)
Angle           | Recall     | Missed (FN)     | False Alarms (FP)  | Specificity
-----------------------------------------------------------------------------------------------
0_deg           | 0.7990     | 3291            | 1181               | 0.9279
0_deg_flip      | 0.7490     | 4111            | 1112               | 0.9322
90_deg          | 0.7457     | 4164            | 1092               | 0.9334
90_deg_flip     | 0.7824     | 3563            | 1240               | 0.9243
180_deg         | 0.7249     | 4505            | 1029               | 0.9372
180_deg_flip    | 0.7659     | 3834            | 1206               | 0.9264
270_deg         | 0.7794     | 3613            | 1214               | 0.9259
270_deg_flip    | 0.7473     | 4138            | 1030               | 0.9372
